# AI Internship – Task 6  
## Image Classification Using Transfer Learning

**Objective**  
Build a high-accuracy binary image classifier (Cats vs Dogs) using **transfer learning** with MobileNetV2.

**Key Details**
- **Base Model**     : MobileNetV2 (ImageNet pretrained)
- **Dataset**        : cats_vs_dogs from TensorFlow Datasets (~25,000 images)
- **Approach**       : Feature extraction → Fine-tuning of top layers
- **Framework**      : TensorFlow / Keras
- **Target**         : Validation accuracy > 94–97%

**Notebook Structure**
1. Setup & Imports
2. Data loading & preprocessing
3. Model creation (transfer learning)
4. Training – Stage 1 (frozen base)
5. Training – Stage 2 (fine-tuning)
6. Evaluation & Visualizations
7. Sample Predictions

In [ ]:
!pip install -q tensorflow-datasets  # usually already there, but safe

import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.optimizers import Adam
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import os

In [ ]:
# Load dataset
(train_ds, val_ds, test_ds), info = tfds.load(
    'cats_vs_dogs',
    split=['train[:80%]', 'train[80%:90%]', 'train[90%:]'],
    with_info=True,
    as_supervised=True
)

IMG_SIZE = 224
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE
NUM_CLASSES = 2  # cat = 0, dog = 1

def preprocess(image, label):
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    image = preprocess_input(image)  # MobileNetV2 expects [-1,1]
    return image, label

train_ds = train_ds.map(preprocess, num_parallel_calls=AUTOTUNE).shuffle(1000).batch(BATCH_SIZE).prefetch(AUTOTUNE)
val_ds   = val_ds.map(preprocess, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)
test_ds  = test_ds.map(preprocess, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)

class_names = ['cat', 'dog']
print("Classes:", class_names)

In [ ]:
IMG_SHAPE = (IMG_SIZE, IMG_SIZE, 3)

base_model = MobileNetV2(input_shape=IMG_SHAPE,
                         include_top=False,
                         weights='imagenet')

# Freeze base at first (feature extraction)
base_model.trainable = False

# Add custom head
model = tf.keras.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(1, activation='sigmoid')  # binary → sigmoid + BinaryCrossentropy
])

model.summary()

In [ ]:
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history_stage1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,           # enough for stage 1
    verbose=1
)

In [ ]:
# Unfreeze last ~30% of layers
base_model.trainable = True
fine_tune_at = len(base_model.layers) // 3 * 2   # ~ last 1/3 layers
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

# Re-compile with lower learning rate
model.compile(
    optimizer=Adam(learning_rate=1e-5),   # very small LR for fine-tuning
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history_stage2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    verbose=1
)

In [ ]:
# Combine histories
acc = history_stage1.history['accuracy'] + history_stage2.history['accuracy']
val_acc = history_stage1.history['val_accuracy'] + history_stage2.history['val_accuracy']
loss = history_stage1.history['loss'] + history_stage2.history['loss']
val_loss = history_stage1.history['val_loss'] + history_stage2.history['val_loss']

epochs_range = range(1, len(acc)+1)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(); plt.title('Accuracy'); plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(); plt.title('Loss'); plt.grid(True)

plt.show()

In [ ]:
test_loss, test_acc = model.evaluate(test_ds)
print(f"\nTest accuracy: {test_acc:.4f}")

# Get predictions for confusion matrix
y_true, y_pred = [], []
for images, labels in test_ds:
    preds = model.predict(images, verbose=0) > 0.5
    y_true.extend(labels.numpy())
    y_pred.extend(preds.flatten().astype(int))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title('Confusion Matrix')
plt.show()

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
def show_predictions(dataset, num=5):
    plt.figure(figsize=(15, 10))
    for images, labels in dataset.take(1):
        preds = model.predict(images)
        for i in range(num):
            plt.subplot(1, num, i+1)
            plt.imshow((images[i].numpy() + 1)/2)  # undo preprocess_input scaling
            pred_label = class_names[int(preds[i] > 0.5)]
            true_label = class_names[labels[i]]
            color = 'green' if pred_label == true_label else 'red'
            plt.title(f"Pred: {pred_label}\nTrue: {true_label}", color=color)
            plt.axis('off')
    plt.suptitle(f"Validation Accuracy: {max(val_acc):.4f}", fontsize=16)
    plt.show()

show_predictions(val_ds)

In [ ]:
model.save('mobilenetv2_cats_vs_dogs.h5')
print("Model saved as mobilenetv2_cats_vs_dogs.h5")